# Environment Setup - E-Commerce Agent Workshop

This notebook sets up all the prerequisites for the E-Commerce Multi-Agent Workshop. We'll provision AWS infrastructure and verify that everything is ready for the hands-on exercises.

## What This Workshop Covers

This workshop demonstrates building a production-ready multi-agent system using:
- **Strands Agent SDK** - Framework for building AI agents
- **Amazon Bedrock** - Managed AI service with Claude models
- **AWS AgentCore** - Production deployment platform
- **DynamoDB** - NoSQL database for orders, accounts, and products

## Infrastructure Overview

The setup script will create:

### 1. DynamoDB Tables
- **ecommerce-workshop-orders** - Customer orders with status tracking
- **ecommerce-workshop-accounts** - Customer account information
- **ecommerce-workshop-products** - Product catalog with inventory

### 2. Sample Data
- Pre-populated orders, accounts, and products for testing
- Realistic e-commerce scenarios

### 3. SSM Parameters
- Resource discovery parameters for agents to find tables
- Configuration management

## Prerequisites

Before running this notebook, ensure you have:
- ✅ AWS credentials configured (via AWS CLI or environment variables)
- ✅ Bedrock model access enabled for Claude Sonnet and Haiku
- ✅ Appropriate IAM permissions for DynamoDB, SSM, and Bedrock

---

## Step 1: Install Python Dependencies

First, we'll install all required Python packages from `requirements.txt`. This includes:
- Strands Agent SDK and evaluation tools
- AWS SDK (boto3) and AgentCore
- OpenTelemetry for observability
- Jupyter and data handling libraries

In [ ]:
# Install all required dependencies
!pip install -r requirements.txt

## Step 2: Provision AWS Infrastructure

Now we'll run the infrastructure setup script. This will:
1. Create three DynamoDB tables with appropriate indexes
2. Load sample data (orders, accounts, products)
3. Create SSM parameters for resource discovery

The script is idempotent - it's safe to run multiple times.

In [ ]:
# Run infrastructure setup
!python setup_infrastructure.py

## Step 3: Verify Infrastructure Setup

Finally, we'll verify that all resources were created successfully. The verification script checks:
- ✅ AWS identity and permissions
- ✅ DynamoDB tables exist and contain data
- ✅ Bedrock model access (Claude Sonnet 4.5 and Haiku 4.5)
- ✅ SSM parameters are configured

If all checks pass, you're ready to proceed with the workshop!

In [ ]:
# Verify infrastructure is ready
!python verify_infrastructure.py

## Step 4: Explore the Sample Dataset

Before diving into building agents, it's helpful to understand the data they'll be working with. The three DynamoDB tables are pre-populated with realistic e-commerce scenarios designed to cover common customer service situations.

### ecommerce-workshop-accounts
Customer profiles with:
- **Membership tiers**: `standard`, `gold`, `platinum` — agents may need to offer tier-appropriate responses
- **Account status**: `active` or `suspended` — agents should check this before processing requests
- **Payment methods and preferences**: realistic multi-field records

### ecommerce-workshop-orders
10 orders covering every major order lifecycle status:
- `pending`, `processing`, `shipped`, `delivered` — the happy path
- `cancelled`, `return_requested`, `refunded` — escalation and resolution flows
- Each order links to a customer via `customer_id` and references products by `product_id`

### ecommerce-workshop-products
Product catalog across 7 categories (Audio, Wearables, Gaming, Monitors, Cameras, Accessories, Furniture) with:
- Stock levels (one product intentionally out of stock)
- Warranties and per-product return policies
- Store-wide policies for returns, shipping, and price matching

Run the cells below to browse the data as DataFrames.

In [ ]:

import json
import textwrap
import pandas as pd
from IPython.display import display

# Load sample data from local files (no AWS connection needed)
with open("sample_data/orders.json") as f:
    orders_data = json.load(f)

with open("sample_data/accounts.json") as f:
    accounts_data = json.load(f)

with open("sample_data/products.json") as f:
    products_data = json.load(f)

print("Sample data loaded successfully!")
print(f"  - {len(accounts_data['accounts'])} customer accounts")
print(f"  - {len(orders_data['orders'])} orders")
print(f"  - {len(products_data['products'])} products in the catalog")


In [ ]:

# Display customer accounts with membership tier breakdown
accounts_df = pd.DataFrame([
    {
        "customer_id": a["customer_id"],
        "name": f"{a['first_name']} {a['last_name']}",
        "membership_tier": a["membership_tier"],
        "account_status": a["account_status"],
        "total_orders": a["total_orders"],
        "total_spent": f"${a['total_spent']:,.2f}",
        "member_since": a["created_date"],
    }
    for a in accounts_data["accounts"]
])

print("=== Customer Accounts ===")
display(accounts_df)

print("\n--- Membership Tier Distribution ---")
tier_counts = accounts_df["membership_tier"].value_counts().rename("count").to_frame()
display(tier_counts)

print("\n--- Account Status Distribution ---")
status_counts = accounts_df["account_status"].value_counts().rename("count").to_frame()
display(status_counts)


In [ ]:

# Display orders with status breakdown
orders_df = pd.DataFrame([
    {
        "order_id": o["order_id"],
        "customer_id": o["customer_id"],
        "status": o["status"],
        "order_date": o["order_date"],
        "items": len(o["items"]),
        "total": f"${o['total']:,.2f}",
    }
    for o in orders_data["orders"]
])

print("=== Orders ===")
display(orders_df)

print("\n--- Order Status Distribution ---")
print("(These represent the different customer service scenarios agents will handle)")
status_counts = orders_df["status"].value_counts().rename("count").to_frame()
display(status_counts)

print("\n--- Orders per Customer ---")
orders_per_customer = (
    orders_df.groupby("customer_id")
    .agg(num_orders=("order_id", "count"))
    .sort_values("num_orders", ascending=False)
)
display(orders_per_customer)


In [ ]:

# Display product catalog and store policies
products_df = pd.DataFrame([
    {
        "product_id": p["product_id"],
        "name": p["name"],
        "category": p["category"],
        "price": f"${p['price']:.2f}",
        "in_stock": "Yes" if p["in_stock"] else "No",
        "stock_qty": p["stock_quantity"],
        "rating": p["rating"],
        "warranty": p["warranty"],
    }
    for p in products_data["products"]
])

print("=== Product Catalog ===")
display(products_df)

print("\n--- Products by Category ---")
category_summary = (
    products_df.assign(price_raw=[p["price"] for p in products_data["products"]])
    .groupby("category")
    .agg(count=("product_id", "count"), avg_price=("price_raw", "mean"))
    .rename(columns={"count": "# Products", "avg_price": "Avg Price ($)"})
    .round(2)
)
display(category_summary)

print("\n=== Store Policies ===")
for key, policy_text in products_data["policies"].items():
    print(f"\n{key.replace('_', ' ').title()}:")
    # Word-wrap at ~100 chars
    import textwrap
    for line in textwrap.wrap(policy_text, width=100):
        print(f"  {line}")


---

## Next Steps

If all verification checks passed, you're ready to start the workshop!

Proceed to:
- **01-multi-agent-prototype** - Build your first multi-agent system
- **02-evaluation-baseline** - Establish evaluation metrics
- **03-production-deployment** - Deploy to AWS AgentCore
- **04-online-eval-observability** - Monitor production agents
- **05-production-batch-evaluation** - Batch evaluation at scale

## Troubleshooting

If verification fails:

### DynamoDB Tables Not Found
```bash
# Re-run setup
python setup_infrastructure.py
```

### Bedrock Model Access Issues
1. Go to AWS Console → Bedrock → Model Access
2. Request access to Claude Sonnet 4.5 and Haiku 4.5
3. Wait for approval (usually instant)

### AWS Credentials Not Configured
```bash
# Configure AWS CLI
aws configure
```

### Clean Up Resources
After completing the workshop, remove all infrastructure:
```bash
python setup_infrastructure.py --cleanup
```

---

**Ready to build intelligent agents? Let's go! 🚀**